# CAPM Colab Parquet Training

Clone repo, read exported Parquet files from Google Drive, preprocess, train models, save each run to Drive under `capm_results/<timestamp>_<model>/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/crypto-autonomous-portfolio-management
!git clone https://github.com/AhmetServet/crypto-autonomous-portfolio-management.git /content/crypto-autonomous-portfolio-management
%cd /content/crypto-autonomous-portfolio-management

In [ ]:
!pip -q install -e . pandas pyarrow numpy scikit-learn xgboost lightgbm torch statsmodels prophet joblib

## Config

Set `DRIVE_DATASET_DIR` to folder containing exported Parquet files, for example files from `db/scripts/export_parquet.py`.

In [ ]:
from pathlib import Path

SYMBOL = 'BTCUSDT'
INTERVAL = '1m'
FORECAST_HORIZON = 15
TARGET_FIELD = 'close'
TARGET_MODE = 'return'

# Change this to your Drive folder.
DRIVE_DATASET_DIR = Path('/content/drive/MyDrive/capm-datasets/btcusdt_1m')
DRIVE_RESULTS_ROOT = Path('/content/drive/MyDrive/capm_results')

# None = use all rows. Use integer for fast experiments, e.g. 300_000.
MAX_ROWS = None

FEATURE_COLUMNS = [
    'bbands_20_2_lower',
    'bbands_20_2_middle',
    'bbands_20_2_upper',
    'ema_20_close',
    'macd_12_26_9_histogram',
    'macd_12_26_9_line',
    'macd_12_26_9_signal',
    'rsi_14_close',
    'sma_20_close',
]

DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('dataset:', DRIVE_DATASET_DIR)
print('results:', DRIVE_RESULTS_ROOT)

## Load Parquet

In [ ]:
import json
import pandas as pd

metadata_path = DRIVE_DATASET_DIR / 'metadata.json'
metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}

ohlcv_file = metadata.get('ohlcv_table', 'coinpair_1_ohlcv') + '.parquet'
feature_file = metadata.get('feature_table', 'coinpair_1_feature') + '.parquet'

ohlcv = pd.read_parquet(DRIVE_DATASET_DIR / ohlcv_file)
features = pd.read_parquet(DRIVE_DATASET_DIR / feature_file)

if MAX_ROWS is not None:
    ohlcv = ohlcv.tail(MAX_ROWS).copy()
    features = features.tail(MAX_ROWS).copy()

print(ohlcv.shape, features.shape)
display(ohlcv.head())
display(features.head())

## Preprocess

Expand JSON feature payload, merge OHLCV + features, create future-return target, temporal split.

In [ ]:
import numpy as np

def parse_payload(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value):
        return {}
    return json.loads(value)

ohlcv = ohlcv[ohlcv['interval'] == INTERVAL].copy()
features = features[features['interval'] == INTERVAL].copy()
if 'is_ready' in features.columns:
    features = features[features['is_ready'] == True].copy()

ohlcv['open_time'] = pd.to_datetime(ohlcv['open_time'], utc=True)
features['open_time'] = pd.to_datetime(features['open_time'], utc=True)

payload_col = 'feature_payload_json' if 'feature_payload_json' in features.columns else 'feature_payload'
payloads = [parse_payload(value) for value in features[payload_col].to_numpy()]
feature_values = pd.DataFrame.from_records(payloads, index=features.index)
feature_values = feature_values.apply(pd.to_numeric, errors='coerce')

features_flat = pd.concat([features[['interval', 'open_time']].copy(), feature_values], axis=1)

df = ohlcv.merge(features_flat, on=['interval', 'open_time'], how='inner')
df = df[df['interval'] == INTERVAL].sort_values('open_time').reset_index(drop=True)

for col in ['open', 'high', 'low', 'close', 'volume', 'quote_asset_volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df['target_value'] = df[TARGET_FIELD].shift(-FORECAST_HORIZON)
if TARGET_MODE == 'return':
    df['target'] = df['target_value'] / df[TARGET_FIELD] - 1.0
else:
    df['target'] = df['target_value']

required_cols = FEATURE_COLUMNS + [TARGET_FIELD, 'target', 'target_value']
df = df.dropna(subset=required_cols).reset_index(drop=True)

n = len(df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df['target']
X_valid = valid_df[FEATURE_COLUMNS]
y_valid = valid_df['target']
X_test = test_df[FEATURE_COLUMNS]
y_test = test_df['target']

print('rows:', n)
print('train/valid/test:', len(train_df), len(valid_df), len(test_df))
print('range:', df['open_time'].min(), '->', df['open_time'].max())

## Shared Training Helpers

In [ ]:
from datetime import datetime, timezone
from time import perf_counter
import joblib
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

def direction_accuracy_from_returns(predicted_return, actual_return):
    return float((np.sign(predicted_return) == np.sign(actual_return)).mean())

def evaluate_predictions(y_true_return, y_pred_return, reference_values):
    actual_values = np.asarray(reference_values) * (1.0 + np.asarray(y_true_return))
    predicted_values = np.asarray(reference_values) * (1.0 + np.asarray(y_pred_return))
    return {
        'rmse': float(np.sqrt(mean_squared_error(actual_values, predicted_values))),
        'mape': float(mean_absolute_percentage_error(actual_values, predicted_values)),
        'direction_accuracy': direction_accuracy_from_returns(np.asarray(y_pred_return), np.asarray(y_true_return)),
    }

def train_eval_save(model_name, model, params):
    started_at = datetime.now(timezone.utc)
    run_id = f"{started_at.strftime('%Y%m%dT%H%M%SZ')}_{SYMBOL.lower()}_{INTERVAL}_{model_name}_{FORECAST_HORIZON}steps"
    run_dir = DRIVE_RESULTS_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    fit_start = perf_counter()
    try:
        model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)
    except TypeError:
        model.fit(X_train, y_train)
    fit_duration = perf_counter() - fit_start

    predict_start = perf_counter()
    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)
    predict_duration = perf_counter() - predict_start

    valid_metrics = evaluate_predictions(y_valid, valid_pred, valid_df[TARGET_FIELD].to_numpy())
    test_metrics = evaluate_predictions(y_test, test_pred, test_df[TARGET_FIELD].to_numpy())

    predictions = pd.DataFrame({
        'open_time': test_df['open_time'].astype(str).to_numpy(),
        'reference_value': test_df[TARGET_FIELD].to_numpy(),
        'actual_return': y_test.to_numpy(),
        'predicted_return': test_pred,
        'actual_value': test_df['target_value'].to_numpy(),
        'predicted_value': test_df[TARGET_FIELD].to_numpy() * (1.0 + test_pred),
    })

    model_path = run_dir / 'model.joblib'
    predictions_path = run_dir / 'predictions.parquet'
    summary_path = run_dir / 'summary.json'
    joblib.dump(model, model_path)
    predictions.to_parquet(predictions_path, index=False)

    summary = {
        'run_id': run_id,
        'model_name': model_name,
        'symbol': SYMBOL,
        'interval': INTERVAL,
        'target_field': TARGET_FIELD,
        'target_mode': TARGET_MODE,
        'forecast_horizon': FORECAST_HORIZON,
        'feature_names': FEATURE_COLUMNS,
        'dataset': {
            'drive_dataset_dir': str(DRIVE_DATASET_DIR),
            'rows': int(len(df)),
            'train_rows': int(len(train_df)),
            'valid_rows': int(len(valid_df)),
            'test_rows': int(len(test_df)),
            'start_time': str(df['open_time'].min()),
            'end_time': str(df['open_time'].max()),
        },
        'params': params,
        'metrics': {
            **test_metrics,
            'fit_duration_seconds': float(fit_duration),
            'predict_duration_seconds': float(predict_duration),
        },
        'validation_metrics': valid_metrics,
        'artifact_paths': {
            'model': str(model_path),
            'predictions': str(predictions_path),
            'summary': str(summary_path),
        },
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary['metrics'], indent=2))
    print('saved:', run_dir)
    return summary

def save_tabular_run(model_name, model, params, fit_duration, predict_duration, valid_pred, test_pred):
    started_at = datetime.now(timezone.utc)
    run_id = f"{started_at.strftime('%Y%m%dT%H%M%SZ')}_{SYMBOL.lower()}_{INTERVAL}_{model_name}_{FORECAST_HORIZON}steps"
    run_dir = DRIVE_RESULTS_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    valid_metrics = evaluate_predictions(y_valid, valid_pred, valid_df[TARGET_FIELD].to_numpy())
    test_metrics = evaluate_predictions(y_test, test_pred, test_df[TARGET_FIELD].to_numpy())
    predictions = pd.DataFrame({
        'open_time': test_df['open_time'].astype(str).to_numpy(),
        'reference_value': test_df[TARGET_FIELD].to_numpy(),
        'actual_return': y_test.to_numpy(),
        'predicted_return': test_pred,
        'actual_value': test_df['target_value'].to_numpy(),
        'predicted_value': test_df[TARGET_FIELD].to_numpy() * (1.0 + test_pred),
    })
    model_path = run_dir / 'model.joblib'
    predictions_path = run_dir / 'predictions.parquet'
    summary_path = run_dir / 'summary.json'
    joblib.dump(model, model_path)
    predictions.to_parquet(predictions_path, index=False)
    summary = {
        'run_id': run_id,
        'model_name': model_name,
        'symbol': SYMBOL,
        'interval': INTERVAL,
        'target_field': TARGET_FIELD,
        'target_mode': TARGET_MODE,
        'forecast_horizon': FORECAST_HORIZON,
        'feature_names': FEATURE_COLUMNS,
        'dataset': {
            'drive_dataset_dir': str(DRIVE_DATASET_DIR),
            'rows': int(len(df)),
            'train_rows': int(len(train_df)),
            'valid_rows': int(len(valid_df)),
            'test_rows': int(len(test_df)),
            'start_time': str(df['open_time'].min()),
            'end_time': str(df['open_time'].max()),
        },
        'params': params,
        'metrics': {**test_metrics, 'fit_duration_seconds': float(fit_duration), 'predict_duration_seconds': float(predict_duration)},
        'validation_metrics': valid_metrics,
        'artifact_paths': {'model': str(model_path), 'predictions': str(predictions_path), 'summary': str(summary_path)},
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary['metrics'], indent=2))
    print('saved:', run_dir)
    return summary

## Train XGBoost

Large CUDA-oriented config. If Colab runtime has no GPU, change `device` to `cpu`.

In [ ]:
from xgboost import XGBRegressor
import cupy as cp

xgb_params = {
    'n_estimators': 8000,
    'max_depth': 10,
    'learning_rate': 0.015,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'colsample_bylevel': 0.85,
    'min_child_weight': 4,
    'reg_alpha': 0.05,
    'reg_lambda': 2.0,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'eval_metric': 'rmse',
    'random_state': 42,
    'n_jobs': -1,
}

X_train_xgb = cp.asarray(X_train.to_numpy(dtype=np.float32))
y_train_xgb = cp.asarray(y_train.to_numpy(dtype=np.float32))
X_valid_xgb = cp.asarray(X_valid.to_numpy(dtype=np.float32))
y_valid_xgb = cp.asarray(y_valid.to_numpy(dtype=np.float32))
X_test_xgb = cp.asarray(X_test.to_numpy(dtype=np.float32))

xgb_model = XGBRegressor(**xgb_params)
fit_start = perf_counter()
xgb_model.fit(X_train_xgb, y_train_xgb, eval_set=[(X_valid_xgb, y_valid_xgb)], verbose=False)
fit_duration = perf_counter() - fit_start

predict_start = perf_counter()
valid_pred = cp.asnumpy(xgb_model.predict(X_valid_xgb))
test_pred = cp.asnumpy(xgb_model.predict(X_test_xgb))
predict_duration = perf_counter() - predict_start

xgb_summary = save_tabular_run('xgboost', xgb_model, xgb_params, fit_duration, predict_duration, valid_pred, test_pred)

## Train LightGBM

GPU-first. If Colab LightGBM wheel lacks GPU support, cell falls back to CPU with clear message.

In [ ]:
from lightgbm import LGBMRegressor

lgbm_params = {
    'n_estimators': 5000,
    'learning_rate': 0.02,
    'num_leaves': 255,
    'max_depth': -1,
    'min_child_samples': 80,
    'subsample': 0.85,
    'subsample_freq': 1,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.05,
    'reg_lambda': 2.0,
    'objective': 'regression',
    'device': 'gpu',
    'gpu_platform_id': 0,
    'gpu_device_id': 0,
    'max_bin': 63,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 1,
}

lgbm_model = LGBMRegressor(**lgbm_params)
fit_start = perf_counter()
try:
    lgbm_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric='rmse')
except Exception as exc:
    if 'GPU Tree Learner was not enabled' not in str(exc) and 'No OpenCL device found' not in str(exc):
        raise
    print('LightGBM GPU unavailable in this Colab runtime, falling back to CPU:', exc)
    lgbm_params = {k: v for k, v in lgbm_params.items() if k not in {'device', 'gpu_platform_id', 'gpu_device_id'}}
    lgbm_params['n_jobs'] = -1
    lgbm_model = LGBMRegressor(**lgbm_params)
    lgbm_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], eval_metric='rmse')
fit_duration = perf_counter() - fit_start

predict_start = perf_counter()
valid_pred = lgbm_model.predict(X_valid)
test_pred = lgbm_model.predict(X_test)
predict_duration = perf_counter() - predict_start

lgbm_summary = save_tabular_run('lightgbm', lgbm_model, lgbm_params, fit_duration, predict_duration, valid_pred, test_pred)

## Sequence Helpers for LSTM / GRU

Uses scaled feature windows and PyTorch. Increase epochs if GPU/time allows.

In [ ]:
import gc
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

gc.collect()
torch.cuda.empty_cache()

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

def make_sequences(frame, feature_cols, target_col, sequence_length):
    values = frame[feature_cols].to_numpy(dtype=np.float32)
    targets = frame[target_col].to_numpy(dtype=np.float32)
    refs = frame[TARGET_FIELD].to_numpy(dtype=np.float32)
    times = frame['open_time'].astype(str).to_numpy()
    actual_values = frame['target_value'].to_numpy(dtype=np.float32)
    X, y, ref, ts, actual = [], [], [], [], []
    for i in range(sequence_length - 1, len(frame)):
        X.append(values[i - sequence_length + 1:i + 1])
        y.append(targets[i])
        ref.append(refs[i])
        ts.append(times[i])
        actual.append(actual_values[i])
    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.float32),
        np.asarray(ref, dtype=np.float32),
        np.asarray(ts),
        np.asarray(actual, dtype=np.float32),
    )

class SequenceModel(nn.Module):
    def __init__(self, model_type, input_size, hidden_size, num_layers, dropout):
        super().__init__()
        rnn_cls = nn.LSTM if model_type == 'lstm' else nn.GRU
        self.rnn = rnn_cls(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, 1),
        )

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

def predict_sequence_batched(model, X, batch_size):
    model.eval()
    preds = []
    loader = DataLoader(
        TensorDataset(torch.tensor(X, dtype=torch.float32)),
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=(DEVICE == 'cuda'),
    )
    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            pred = model(xb).detach().cpu().numpy()
            preds.append(pred)
    return np.concatenate(preds)

def train_eval_save_sequence(model_name, params):
    gc.collect()
    torch.cuda.empty_cache()

    started_at = datetime.now(timezone.utc)
    run_id = f"{started_at.strftime('%Y%m%dT%H%M%SZ')}_{SYMBOL.lower()}_{INTERVAL}_{model_name}_{FORECAST_HORIZON}steps"
    run_dir = DRIVE_RESULTS_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    sequence_length = params['sequence_length']
    scaler = StandardScaler()
    scaler.fit(train_df[FEATURE_COLUMNS])

    train_scaled = train_df.copy()
    valid_scaled = valid_df.copy()
    test_scaled = test_df.copy()
    train_scaled[FEATURE_COLUMNS] = scaler.transform(train_df[FEATURE_COLUMNS])
    valid_scaled[FEATURE_COLUMNS] = scaler.transform(valid_df[FEATURE_COLUMNS])
    test_scaled[FEATURE_COLUMNS] = scaler.transform(test_df[FEATURE_COLUMNS])

    Xtr, ytr, _, _, _ = make_sequences(train_scaled, FEATURE_COLUMNS, 'target', sequence_length)
    Xva, yva, ref_va, _, _ = make_sequences(valid_scaled, FEATURE_COLUMNS, 'target', sequence_length)
    Xte, yte, ref_te, ts_te, actual_te = make_sequences(test_scaled, FEATURE_COLUMNS, 'target', sequence_length)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.float32)),
        batch_size=params['batch_size'],
        shuffle=True,
        drop_last=False,
        pin_memory=(DEVICE == 'cuda'),
    )
    model = SequenceModel(
        model_type=model_name,
        input_size=len(FEATURE_COLUMNS),
        hidden_size=params['hidden_size'],
        num_layers=params['num_layers'],
        dropout=params['dropout'],
    ).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['learning_rate'], weight_decay=params['weight_decay'])
    loss_fn = nn.HuberLoss(delta=params['huber_delta'])

    fit_start = perf_counter()
    best_valid = float('inf')
    best_state = None
    patience_left = params['patience']
    for epoch in range(params['epochs']):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), params['grad_clip'])
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))
        valid_pred = predict_sequence_batched(model, Xva, params['eval_batch_size'])
        valid_rmse = float(np.sqrt(mean_squared_error(yva, valid_pred)))
        if valid_rmse < best_valid:
            best_valid = valid_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = params['patience']
        else:
            patience_left -= 1
        print(f"epoch={epoch + 1} train_loss={np.mean(train_losses):.8f} valid_return_rmse={valid_rmse:.8f}")
        if patience_left <= 0:
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    fit_duration = perf_counter() - fit_start

    predict_start = perf_counter()
    valid_pred = predict_sequence_batched(model, Xva, params['eval_batch_size'])
    test_pred = predict_sequence_batched(model, Xte, params['eval_batch_size'])
    predict_duration = perf_counter() - predict_start

    valid_metrics = evaluate_predictions(yva, valid_pred, ref_va)
    test_metrics = evaluate_predictions(yte, test_pred, ref_te)
    predictions = pd.DataFrame({
        'open_time': ts_te,
        'reference_value': ref_te,
        'actual_return': yte,
        'predicted_return': test_pred,
        'actual_value': actual_te,
        'predicted_value': ref_te * (1.0 + test_pred),
    })

    model_path = run_dir / 'model.pt'
    scaler_path = run_dir / 'scaler.joblib'
    predictions_path = run_dir / 'predictions.parquet'
    summary_path = run_dir / 'summary.json'
    torch.save({'model_state_dict': model.state_dict(), 'params': params, 'feature_names': FEATURE_COLUMNS}, model_path)
    joblib.dump(scaler, scaler_path)
    predictions.to_parquet(predictions_path, index=False)

    summary = {
        'run_id': run_id,
        'model_name': model_name,
        'symbol': SYMBOL,
        'interval': INTERVAL,
        'target_field': TARGET_FIELD,
        'target_mode': TARGET_MODE,
        'forecast_horizon': FORECAST_HORIZON,
        'sequence_length': sequence_length,
        'feature_names': FEATURE_COLUMNS,
        'dataset': {'drive_dataset_dir': str(DRIVE_DATASET_DIR), 'rows': int(len(df)), 'train_rows': int(len(train_df)), 'valid_rows': int(len(valid_df)), 'test_rows': int(len(test_df))},
        'params': params,
        'metrics': {**test_metrics, 'fit_duration_seconds': float(fit_duration), 'predict_duration_seconds': float(predict_duration)},
        'validation_metrics': valid_metrics,
        'artifact_paths': {'model': str(model_path), 'scaler': str(scaler_path), 'predictions': str(predictions_path), 'summary': str(summary_path)},
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary['metrics'], indent=2))
    print('saved:', run_dir)

    del Xtr, ytr, Xva, yva, Xte, yte
    gc.collect()
    torch.cuda.empty_cache()

    return summary

## Train LSTM

In [ ]:
lstm_params = {
    'sequence_length': 60,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.20,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 512,
    'eval_batch_size': 2048,
    'epochs': 40,
    'patience': 6,
    'grad_clip': 1.0,
    'huber_delta': 0.002,
}

lstm_summary = train_eval_save_sequence('lstm', lstm_params)

## Train GRU

In [ ]:
gru_params = {
    'sequence_length': 60,
    'hidden_size': 128,
    'num_layers': 2,
    'dropout': 0.20,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'batch_size': 512,
    'eval_batch_size': 2048,
    'epochs': 40,
    'patience': 6,
    'grad_clip': 1.0,
    'huber_delta': 0.002,
}

gru_summary = train_eval_save_sequence('gru', gru_params)

## Statistical Helpers for ARIMA / Prophet

These are univariate close-price models. Use fewer rows first; full 1m history can be slow.

In [ ]:
def save_statistical_run(model_name, params, fit_duration, predict_duration, test_pred_values):
    started_at = datetime.now(timezone.utc)
    run_id = f"{started_at.strftime('%Y%m%dT%H%M%SZ')}_{SYMBOL.lower()}_{INTERVAL}_{model_name}_{FORECAST_HORIZON}steps"
    run_dir = DRIVE_RESULTS_ROOT / run_id
    run_dir.mkdir(parents=True, exist_ok=True)
    reference = test_df[TARGET_FIELD].to_numpy()
    actual = test_df['target_value'].to_numpy()
    predicted = np.asarray(test_pred_values, dtype=float)
    actual_return = actual / reference - 1.0
    predicted_return = predicted / reference - 1.0
    metrics = {
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'mape': float(mean_absolute_percentage_error(actual, predicted)),
        'direction_accuracy': direction_accuracy_from_returns(predicted_return, actual_return),
        'fit_duration_seconds': float(fit_duration),
        'predict_duration_seconds': float(predict_duration),
    }
    predictions = pd.DataFrame({
        'open_time': test_df['open_time'].astype(str).to_numpy(),
        'reference_value': reference,
        'actual_return': actual_return,
        'predicted_return': predicted_return,
        'actual_value': actual,
        'predicted_value': predicted,
    })
    predictions_path = run_dir / 'predictions.parquet'
    summary_path = run_dir / 'summary.json'
    predictions.to_parquet(predictions_path, index=False)
    summary = {
        'run_id': run_id,
        'model_name': model_name,
        'symbol': SYMBOL,
        'interval': INTERVAL,
        'target_field': TARGET_FIELD,
        'target_mode': 'price',
        'forecast_horizon': FORECAST_HORIZON,
        'feature_names': [],
        'dataset': {'drive_dataset_dir': str(DRIVE_DATASET_DIR), 'rows': int(len(df)), 'train_rows': int(len(train_df)), 'valid_rows': int(len(valid_df)), 'test_rows': int(len(test_df))},
        'params': params,
        'metrics': metrics,
        'artifact_paths': {'predictions': str(predictions_path), 'summary': str(summary_path)},
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(metrics, indent=2))
    print('saved:', run_dir)
    return summary

## Train ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

# Full 1m ARIMA can be expensive. Increase ARIMA_TRAIN_TAIL after first smoke run.
arima_params = {'order': (5, 1, 2), 'train_tail_rows': 200_000}

series = pd.concat([train_df[TARGET_FIELD], valid_df[TARGET_FIELD]]).tail(arima_params['train_tail_rows']).astype(float)
fit_start = perf_counter()
arima_fit = ARIMA(series, order=arima_params['order']).fit()
fit_duration = perf_counter() - fit_start

predict_start = perf_counter()
arima_pred = arima_fit.forecast(steps=len(test_df) + FORECAST_HORIZON).to_numpy()[FORECAST_HORIZON:FORECAST_HORIZON + len(test_df)]
predict_duration = perf_counter() - predict_start

arima_summary = save_statistical_run('arima', arima_params, fit_duration, predict_duration, arima_pred)

## Train Prophet

In [ ]:
from prophet import Prophet

# Prophet on millions of 1m rows can be slow. Increase train_tail_rows after first run.
prophet_params = {
    'train_tail_rows': 200_000,
    'daily_seasonality': True,
    'weekly_seasonality': True,
    'yearly_seasonality': False,
    'changepoint_prior_scale': 0.05,
    'seasonality_prior_scale': 10.0,
}

prophet_train = pd.concat([train_df, valid_df]).tail(prophet_params['train_tail_rows'])[['open_time', TARGET_FIELD]].copy()
prophet_train['ds'] = prophet_train['open_time'].dt.tz_convert(None)
prophet_train['y'] = prophet_train[TARGET_FIELD].astype(float)

fit_start = perf_counter()
prophet_model = Prophet(
    daily_seasonality=prophet_params['daily_seasonality'],
    weekly_seasonality=prophet_params['weekly_seasonality'],
    yearly_seasonality=prophet_params['yearly_seasonality'],
    changepoint_prior_scale=prophet_params['changepoint_prior_scale'],
    seasonality_prior_scale=prophet_params['seasonality_prior_scale'],
)
prophet_model.fit(prophet_train[['ds', 'y']])
fit_duration = perf_counter() - fit_start

future = pd.DataFrame({'ds': (test_df['open_time'] + pd.to_timedelta(FORECAST_HORIZON, unit='min')).dt.tz_convert(None)})
predict_start = perf_counter()
prophet_pred = prophet_model.predict(future)['yhat'].to_numpy()
predict_duration = perf_counter() - predict_start

prophet_summary = save_statistical_run('prophet', prophet_params, fit_duration, predict_duration, prophet_pred)

## Compare Runs

In [ ]:
import glob

summaries = []
for path in glob.glob(str(DRIVE_RESULTS_ROOT / '*' / 'summary.json')):
    payload = json.loads(Path(path).read_text())
    row = {
        'run_id': payload['run_id'],
        'model_name': payload['model_name'],
        **payload['metrics'],
        'summary_path': path,
    }
    summaries.append(row)

leaderboard = pd.DataFrame(summaries).sort_values(['direction_accuracy', 'rmse'], ascending=[False, True])
display(leaderboard)